# AWS Cloud Analytics Pipeline: NYC Taxi Trip Data

Starter notebook. Run `download_data.py` first and set up the Glue crawler / Athena database before running the Athena cells below.

See `README.md` for the full setup steps.

In [ ]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt

PROFILE_NAME = "preethi-portfolio"
REGION = "us-east-2"

session = boto3.Session(profile_name=PROFILE_NAME, region_name=REGION)
print("Authenticated as:", session.client("sts").get_caller_identity()["Arn"])

## Step 1: Quick local sanity check

Load the downloaded Parquet file directly with pandas before setting up Glue/Athena, just to confirm the data looks right.

In [ ]:
LOCAL_FILE = "yellow_tripdata_2026-05.parquet"  # update to match download_data.py

df = pd.read_parquet(LOCAL_FILE)
print(df.shape)
df.head()

## Step 2: Query via Athena

Fill in `DATABASE` and `TABLE` with whatever the Glue crawler actually named them, and `ATHENA_RESULTS_BUCKET` with an S3 location Athena can write query results to (Athena requires this).

TODO: replace/extend the example query below once you've explored the schema — this is just a starting point, not a finished analysis.

In [ ]:
import time

athena = session.client("athena")

DATABASE = "CHANGE-ME"  # database name created by the Glue crawler
TABLE = "CHANGE-ME"     # table name created by the Glue crawler
ATHENA_RESULTS_BUCKET = "s3://CHANGE-ME-your-bucket/athena-results/"


def run_athena_query(sql):
    exec_id = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE},
        ResultConfiguration={"OutputLocation": ATHENA_RESULTS_BUCKET},
    )["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]["Status"]["State"]
        if status in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(2)

    if status != "SUCCEEDED":
        raise RuntimeError(f"Athena query {status}")

    result_path = f"{ATHENA_RESULTS_BUCKET}{exec_id}.csv"
    return pd.read_csv(result_path)


# Example starter query -- adjust column names once you've checked the actual
# schema the crawler produced (Parquet column names may differ slightly).
example_sql = f"""
SELECT
    HOUR(tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trip_count,
    AVG(date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime)) AS avg_trip_duration_sec
FROM {TABLE}
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY pickup_hour
"""

# result_df = run_athena_query(example_sql)
# result_df

## Step 3: Visualize

TODO: once `result_df` above has real query results, plot trip volume / duration by hour here.

In [ ]:
# result_df.plot(x="pickup_hour", y="trip_count", kind="bar", figsize=(10, 5), title="Trips by Pickup Hour")
# plt.show()